# Task 14 – Interactive LLM Web Application using Streamlit and Ollama

## Task Title

Developing an Interactive Web-Based Multi-PDF Question Answering Application (Knowledge Lens AI)

---

## Objective

- Deploy the PDF Question Answering RAG system built in Task 13 as an interactive web application using Streamlit.
- Integrate Streamlit with LangChain, ChromaDB, Sentence Transformers, and Ollama (Gemma).
- Enable users to upload one or multiple PDF documents with strict `.pdf` file validation.
- Display uploaded document filenames, active document count, and combined total page count.
- Allow seamless switching and simultaneous querying across multiple PDF documents.
- Implement multi-session conversation history, adaptive response length controls (`Auto`, `Concise`, `Detailed`), response generation timers, and expandable source chunk transparency.
- Automatically log all user queries, generated answers, timestamps, and response times into CSV files.

---

## Folder Structure

```text
STREAMLIT LLM APPLICATION/
├── .streamlit/
│   └── config.toml
├── assets/
│   ├── logo.png
│   └── sparkle.png
├── chroma_db/
├── dataset/
│   ├── uploaded_files/
│   ├── answers.csv
│   ├── queries.csv
│   └── sample.pdf
├── notebook/
│   └── Task14_Streamlit_LLM_App.ipynb
├── screenshots/
│   ├── 01_home_interface.png
│   ├── 02_sidebar_controls.png
│   ├── 03_multiturn_psychology_qa.png
│   ├── 04_healthcare_qa_history.png
│   ├── 05_cross_document_synthesis.png
│   ├── 06_retrieved_source_chunks.png
│   └── 07_csv_logging_output.png
├── app.py
├── pdf_qa.py
├── README.md
└── requirements.txt
```

---

## PDF Selection & Multi-Document Knowledge Base

Two primary domain reports were uploaded and indexed into persistent ChromaDB collections to test single-document precision and cross-document synthesis:

1. **AI in Healthcare Report (`AI in Healthcare Report .pdf`)**
   - Covers Machine Learning, Deep Learning, Medical Imaging (`IDx-DR`), Drug Discovery (`AlphaFold`), Predictive Analytics, Telemedicine, Regulatory Frameworks (`HIPAA`, `GDPR`, `FDA SaMD / TPLC`), and Federated Learning.

2. **The Psychology of Binge-Watching and the Role of AI (`The Psychology of Binge-Watching and the Role of AI.pdf`)**
   - Covers cognitive and behavioral mechanisms (`Zeigarnik Effect`, dopamine reward loops), AI recommendation algorithms, predictive behavioral modeling, explainable AI, and on-device Federated Learning.

Supporting multiple PDFs simultaneously allowed the web application to synthesize comparative answers across distinct domains in a single turn.

---

## Text Extraction & File Validation

Uploaded files are validated in `app.py` to ensure only genuine `.pdf` files (`type=["pdf"]` and MIME type `application/pdf`) are accepted and saved inside `dataset/uploaded_files/`.

Text extraction in `pdf_qa.py` (`load_pdf`) uses `pypdf.PdfReader` to extract text page-by-page while embedding explicit page markers (`--- Page X ---`) and calculating the total page count displayed in the active document banner.

---

## Document Chunking

Extracted document text is divided into overlapping contextual segments using `RecursiveCharacterTextSplitter` in `pdf_qa.py`:

Configuration:

- Chunk Size: 1100 characters
- Chunk Overlap: 250 characters
- Separators: `["\n\n", "\n", ". ", " ", ""]`

Preserving page markers and 250-character overlap ensures technical definitions, clinical entities, and regulatory acronyms remain intact across chunk boundaries.

---

## Embedding Generation

The Sentence Transformer model:

`all-MiniLM-L6-v2`

was used to encode every document chunk and user sub-query into normalized dense vector embeddings.

Normalization enables fast dot-product cosine similarity scoring alongside keyword and entity weighting.

---

## Persistent ChromaDB Collections

Each uploaded PDF is indexed into its own sanitized, persistent ChromaDB collection inside `chroma_db/` (prefixed with `klens_`) storing:

- Document Chunk Text
- Dense Vector Embeddings
- Source Metadata (`source` filename and `chunk_id`)

Session caching (`st.session_state.collections_cache`) prevents redundant re-indexing when switching between active PDFs in the sidebar.

---

## Loading Ollama and Gemma

The local Gemma model is connected via `langchain_ollama.OllamaLLM` with deterministic and clean-streaming controls:

- `model="gemma"`
- `num_ctx=4096`
- `temperature=0.0`
- `num_predict`: Dynamically adjusted by response mode (`200` for Concise, `280–320` for Auto, `450` for Detailed)
- `stop=["<start_of_turn>", "</start_of_turn>", "<end_of_turn>", "</end_of_turn>"]`
- `keep_alive="15m"`

Explicit turn-boundary stop tokens prevent Gemma's internal template tokens from appearing at the end of streamed responses.

---

## Hybrid Retrieval-Augmented Generation (RAG) Workflow

The web application executes an enhanced multi-stage RAG pipeline:

User Question (via Streamlit Chat Input)

↓

Sub-Query Decomposition & Keyword Extraction

↓

Multi-Collection Hybrid Scoring (Semantic Cosine Similarity + IDF Keyword Weights + Entity/Phrase Boosts − TOC/References Penalty)

↓

Top-8 Ranked Chunks Across Active PDFs

↓

Prompt Construction (Context + Recent Conversation History + Response Length Guidelines)

↓

Real-Time Token Streaming (`st.write_stream`) via Ollama Gemma

↓

Streamed Web Answer + Generation Timer + Expandable Source Chunks + CSV Logging

---

## Interactive Web Interface Features (`app.py`)

The Streamlit frontend (**Knowledge Lens AI**) enhances usability and transparency through:

- **Custom Branding & Theme:** Deep midnight-violet frame (`#1D142C`), warm ivory reading container (`#FFFDF9`), custom sidebar logo (`assets/logo.png`), and custom assistant avatar (`assets/sparkle.png`).
- **Sidebar Document Controls:** Multi-PDF uploader with strict validation, active document multi-select switcher (`st.multiselect`), and `Response Length` selector (`Auto`, `Concise`, `Detailed`).
- **Active Document Status Card:** Displays active PDF filenames and combined total page count.
- **Multi-Session Chat History:** UUID-backed conversation threads with `New Chat`, `Clear All`, and clickable saved chat history buttons in the sidebar.
- **Transparency & Logging:** Displays response generation time (`Generated in X sec`), an expandable `View Retrieved Source Chunks` panel under every answer, and automatic logging to `dataset/queries.csv` and `dataset/answers.csv`.

---

## Sample Questions Tested

1. What was IDx-DR, and how does DeepMind's AlphaFold accelerate drug discovery?
2. What regulatory frameworks and privacy laws govern AI systems like these in healthcare?
3. What is predictive analytics in healthcare, and how does it help hospitals prevent readmissions and sepsis?
4. How does the Zeigarnik Effect and dopamine reward loop drive binge-watching behavior on streaming platforms?
5. Compare how both reports use Federated Learning to protect user or patient privacy.
6. How do both reports address "black-box" AI and explainability to maintain human trust and oversight?

---

## Sample Output

| Question | Active PDF(s) | Answer Summary |
| :--- | :--- | :--- |
| What is predictive analytics in healthcare, and how does it help hospitals prevent readmissions and sepsis? | AI in Healthcare Report .pdf | Predictive analytics uses machine learning on vitals, lab results, and EHR data to calculate continuous risk scores, enabling early sepsis detection and proactive interventions to prevent readmissions. |
| How does the Zeigarnik Effect and dopamine reward loop drive binge-watching behavior on streaming platforms? | The Psychology of Binge-Watching and the Role of AI.pdf | The Zeigarnik Effect creates cognitive tension from unresolved episode cliffhangers, while dopamine release from plot twists and emotional payoffs sustains high neurological arousal across consecutive episodes. |
| How do both reports address "black-box" AI and explainability to maintain human trust and oversight? | Both PDFs | Healthcare AI emphasizes Explainable AI (XAI) to mitigate deep learning opacity, while streaming AI introduces interactive preference sliders and natural-language controls to restore viewer agency and transparency. |

---

## Captured Application Screenshots (`screenshots/`)

1. **`01_home_interface.png` — Home Interface & Branding:** Displays the custom title, sidebar logo, PDF uploader, and welcome screen.
2. **`02_sidebar_controls.png` — Sidebar Controls & History Panel:** Displays the `Response Length` selector (`Auto`, `Concise`, `Detailed`) and `New Chat` / `Clear All` session buttons.
3. **`03_multiturn_psychology_qa.png` — Multi-Turn Conversation & Timer:** Demonstrates follow-up querying on Binge-Watching Psychology with the response generation timer (`Generated in 168.91 sec`).
4. **`04_healthcare_qa_history.png` — Multi-PDF Selection & Healthcare QA:** Shows both PDFs active in the sidebar, the Healthcare Predictive Analytics response, and saved chat history.
5. **`05_cross_document_synthesis.png` — Cross-Document Synthesis:** Demonstrates a unified comparative answer synthesizing Explainable AI across both reports.
6. **`06_retrieved_source_chunks.png` — Retrieved Source Chunks:** Shows the expanded `View Retrieved Source Chunks` panel displaying exact source citations (`Source: AI in Healthcare Report .pdf`).
7. **`07_csv_logging_output.png` — Automated CSV Logging:** Displays logged queries, answers, timestamps, and response metrics recorded automatically in `dataset/answers.csv`.

---

## Retrieved Context & Source Transparency

Below every generated response in the Streamlit web interface:

- An expandable **View Retrieved Source Chunks** section displays the exact retrieved chunks along with their source PDF filename and page marker.
- A generation timer displays the exact inference latency (`Generated in X sec`).
- Users can verify every claim directly against the underlying PDF excerpts, eliminating black-box responses.

---

## Comparison: CLI Chatbot (Task 13) vs Streamlit Web App (Task 14)

### Task 13 – Command-Line PDF QA
- Single PDF file configured in code.
- Terminal-only text input and output.
- Basic vector similarity retrieval.
- Single linear session without visual source inspection.

### Task 14 – Streamlit Web Application (Knowledge Lens AI)
- Interactive browser UI with multi-PDF upload, validation, and dynamic switching.
- Hybrid retrieval combining semantic embeddings, sub-query decomposition, IDF weights, and entity boosting.
- Real-time streamed responses, generation latency metrics, and expandable source chunks.
- Persistent multi-session chat history (`New Chat` / `Clear All`) and automated CSV logging.

### Observation
Deploying the RAG pipeline in Streamlit transformed a backend script into an accessible, transparent, and multi-document research assistant suitable for end users.

---

## Observations

- Streamlit provided a responsive, browser-based interface for real-time PDF question answering.
- Multi-PDF indexing in ChromaDB enabled seamless switching and cross-document synthesis across unrelated domains.
- Hybrid scoring (semantic similarity + IDF + entity boosts) significantly improved retrieval accuracy for technical acronyms and multi-part questions.
- Turn-boundary stop tokens ensured clean, well-formatted streamed responses from Gemma.
- Automated CSV logging (`queries.csv` and `answers.csv`) preserved a complete audit trail of evaluation runs.

---

## Strengths

- User-friendly, custom-styled web interface with intuitive sidebar controls.
- Strict PDF upload validation and multi-document cross-synthesis.
- Grounded, explainable answers with expandable source chunk citations.
- Real-time response streaming and per-query latency tracking.
- Completely local, privacy-preserving execution using Ollama and ChromaDB.

---

## Limitations

- Local CPU/GPU inference time increases when generating long `Detailed` answers across multiple large PDFs.
- Scanned image-only PDFs without embedded text require OCR preprocessing before extraction.
- Very large document uploads require initial embedding time before the first query.

---

## Future Work

- Add OCR support (`pytesseract` / `pdf2image`) for scanned PDFs.
- Add downloadable chat transcripts (Markdown / PDF export) directly from the sidebar.
- Add a model selector dropdown in the sidebar to switch between local Ollama models (`gemma`, `llama3`, `mistral`).
- Package the complete application with an evaluation dashboard for the Week 15 Mini Capstone Project.

---

## Conclusion

An interactive, browser-based PDF Question Answering application (**Knowledge Lens AI**) was successfully developed and deployed using **Streamlit**, **LangChain**, **ChromaDB**, **Sentence Transformers**, and **Ollama (Gemma)**. By combining multi-PDF upload and validation, hybrid semantic-keyword retrieval, real-time token streaming, multi-session chat history, and transparent source chunk inspection, the application fulfills all requirements of Week 14 and establishes a strong foundation for the Week 15 Mini Capstone Project.